# OpenAlex from R: green and circular-economy science in five blocks

**DAISY 2026 summer school, Taranto. Session "Publications as innovation data: measuring (green) science with OpenAlex, at scale with BigQuery"** (Massimiliano Coda Zabetta)

Leg B of the demo: the questions we ask VOSviewer (leg A) and BigQuery (leg C), asked to the OpenAlex REST API from R with the `openalexR` package (Aria, Le, Cuccurullo, Belfiore and Choe 2024, *R Journal* 15(4)). A Python twin with `pyalex` (`daisy_openalex_api_python.ipynb`) has the same blocks.

Blocks: **B0** anatomy of a record and cost, **B1** trend, **B2** geography and RTA, **B3** SDG landscape, **B4** extras (not demoed).

Each block has: **concept** (what and why), **code** (one comment per line, with the Stata equivalent where one exists), **interpret** (what to say about the output), **exercise** (one line to try after the session).

Runtime: Colab, menu Runtime > Change runtime type > R. Cells run top to bottom; nothing is loaded from disk.

## Setup: packages and API key

Since 13 February 2026 the OpenAlex API expects a key for anything beyond light demo use. The key is free: log in at [openalex.org/settings/api](https://openalex.org/settings/api), copy it. A free key gives 1 USD of usage per day; a list, filter or group_by call costs 0.0001 USD, a text search 0.001 USD, a single-record lookup nothing. Keyless calls still work with a 0.10 USD daily budget (about 1,000 calls), enough to run this notebook once. Every response carries `meta.cost_usd`, so the price of a call is visible.

This notebook makes about 25 calls, roughly 0.003 USD with a key. Paste the key in the cell below; if you leave the placeholder, the calls run keyless.

`openalexR` 3.x follows the current OpenAlex (Walden) schema; version 2.0.0 fails when it parses works. `install.packages()` gets the CRAN version (3.1.0, July 2026). Where an argument differs between 2.0.0 and 3.x the code comments say so.

In [ ]:
# Install openalexR from CRAN (about 20 seconds in Colab; needs >= 3.0.0 for the current OpenAlex schema)
install.packages("openalexR", quiet = TRUE)
# dplyr, tidyr, ggplot2 come preinstalled in the Colab R runtime; install them only if missing (local machines)
for (p in c("dplyr", "tidyr", "ggplot2")) if (!requireNamespace(p, quietly = TRUE)) install.packages(p, quiet = TRUE)

In [ ]:
library(openalexR)   # talks to the OpenAlex API
library(dplyr)       # filter, select, mutate, join, summarise (Stata: keep, gen, merge, collapse)
library(tidyr)       # unnest list-columns (Stata: reshape long)
library(ggplot2)     # charts
packageVersion("openalexR")   # should print 3.1.0 or later

In [ ]:
my_key <- "PASTE-YOUR-KEY"                                          # paste your key between the quotes (openalex.org/settings/api)
if (my_key != "PASTE-YOUR-KEY") options(openalexR.apikey = my_key)   # register it once for the session; with the placeholder the calls run keyless
options(openalexR.mailto = "you@example.org")                       # optional courtesy address (OpenAlex "polite pool")

## B0. Anatomy of one record

**Concept.** One OpenAlex work is a nested JSON record: flat fields (title, year, type, citations, FWCI), then lists inside it: authorships (each with author, position and one or more institutions with a country), topics (each with subfield, field, domain and a score; the first one is the `primary_topic`), keywords, SDG tags, referenced works. Our example is the review by Ghisellini, Cialani and Ulgiati (2016, *Journal of Cleaner Production*), OpenAlex id W1732240353, found with a free lookup by DOI (10.1016/j.jclepro.2015.09.007). `oa_fetch()` turns the record into one tibble row with list-columns; `unnest()` opens them (Stata: `reshape long`).

In [ ]:
# Fetch one work by its OpenAlex id (single lookups are free; a DOI works too: identifier = "https://doi.org/10.1016/j.jclepro.2015.09.007")
paper <- oa_fetch(entity = "works", identifier = "W1732240353")
# Flat fields: one row (Stata: list title year type citations in 1)
paper %>% select(display_name, publication_year, type, source_display_name, cited_by_count, fwci, is_oa)

In [ ]:
# Topics: a long table, four rows per topic (type = topic, subfield, field, domain); i = 1 is the primary topic
paper$topics[[1]]

In [ ]:
# SDG tags: NA here, the classifier gave this circular-economy review no SDG (keep this in mind for B3)
paper$sustainable_development_goals
# Keywords: algorithmic, with a score
head(paper$keywords[[1]], 5)

In [ ]:
# Authorships: one row per author, with a nested affiliations table; unnest it to get institution and country
# (Stata: reshape long twice, authors then affiliations)
paper$authorships[[1]] %>%
  select(display_name, author_position, affiliations) %>%
  unnest(affiliations, names_sep = "_") %>%
  select(display_name, author_position, affiliations_display_name, affiliations_country_code)

In [ ]:
# The price of a call: count_only = TRUE returns the meta block only (count and cost_usd), no records
meta <- oa_fetch(entity = "works", primary_topic.id = "T10471", count_only = TRUE)
meta$count      # works whose primary topic is T10471, Climate Change Policy and Economics (about 105,000 in Aug 2026)
meta$cost_usd   # 0.0001 USD for a filter call

**Interpret.** Year 2015, not 2016: OpenAlex records the earliest publication date (online first). Three authors, four author-institution rows, two countries for one author: counting by country requires a decision (whole or fractional). The primary topic is T10539 "Sustainable Supply Chain Management" (subfield Strategy and Management), the closest OpenAlex topic to "circular economy"; no topic is named circular economy, which is why B1 and B2 use a set of topics. The SDG list is empty: a classifier assigns SDG tags, and it misses this paper (SDG 12 would be the natural tag).

**Exercise.** Replace the identifier with the DOI of your own paper (or Kirchherr, Reike and Hekkert 2017, W2756283300) and check its topics and SDGs.

## B1. Trend: is climate and circular-economy science growing faster than science?

**Concept.** `group_by` asks the server to count works by a field and to send back only the small table of counts (Stata: `collapse (count), by(year)`, done on OpenAlex's side). One call gives the numerator (works in a topic, by year), one call the denominator (all works, by year); the share tells whether the field grows faster than the total. Topic T10471 is "Climate Change Policy and Economics" (about 105,000 works, 69,000 of them in 2010-2025). For circular economy we pass a set of topic ids as an OR list. In the URL that list reads `primary_topic.id:T10539|T12746|...`; we build the string once (`ce_or`) and pass it to `oa_fetch()`. A character vector would work too, but openalexR splits vectors longer than 50 values into several calls and stacks the results, which would duplicate the `group_by` keys if the set ever grows past 50.

In [ ]:
# Works whose primary topic is T10471, counted by publication year on the server (Stata: collapse (count) id, by(year))
cc_year <- oa_fetch(entity = "works", primary_topic.id = "T10471", publication_year = "2010-2025", group_by = "publication_year")
# All works by year, no topic filter: the denominator
all_year <- oa_fetch(entity = "works", publication_year = "2010-2025", group_by = "publication_year")
# Result columns: key (the year, as text), key_display_name, count
head(cc_year, 3)

In [ ]:
# Circular-economy topic set: the 28 topics of session/data/ce_topic_ids_api.txt (names, work counts and pruning notes in session/data/README.md);
# the same list is inlined in the SQL of leg C and pasted in the VOSviewer URL of leg A, so the three legs answer on the same set.
# The set is a choice (T10539 Sustainable Supply Chain Management, T12017 Recycling and Waste Management Techniques, T11091 Extraction and Separation
# Processes for battery recycling, T13180 green chemistry, ...); prune it here and every number in B1 and B2 changes.
ce_topics <- c(
               "T10171", "T10264", "T10284", "T10435",
               "T10539", "T10753", "T11091", "T11108",
               "T11275", "T11672", "T11781", "T11847",
               "T12017", "T12118", "T12186", "T12746",
               "T12774", "T12838", "T12920", "T13045",
               "T13140", "T13180", "T13240", "T13477",
               "T13790", "T14138", "T14164", "T14179")
length(ce_topics)   # 28
# One string "T10171|T10264|..." for the filter (the API accepts up to 100 ids per filter). Passing the vector itself also works,
# but openalexR splits vectors longer than 50 into several calls and stacks the results, which would duplicate group_by keys.
ce_or <- paste(ce_topics, collapse = "|")
# CE works by year: the same group_by call with the OR list as the topic filter (primary_topic.id:T10171|T10264|...)
ce_year <- oa_fetch(entity = "works", primary_topic.id = ce_or, publication_year = "2010-2025", group_by = "publication_year")

In [ ]:
# Merge the three tables on year (Stata: merge 1:1 year) and compute shares of world output
trend <- all_year %>% transmute(year = as.integer(key), all = count) %>%          # denominator: key (text) -> integer year
  inner_join(cc_year %>% transmute(year = as.integer(key), climate = count), by = "year") %>%    # add T10471 counts
  inner_join(ce_year %>% transmute(year = as.integer(key), circular = count), by = "year") %>%   # add CE counts
  mutate(climate_share = 100 * climate / all,      # share of world output, in percent
         circular_share = 100 * circular / all) %>%
  arrange(year)                                    # chronological order
trend                                              # print the table (Stata: list)

In [ ]:
# Long format for ggplot (Stata: reshape long share, i(year) j(series)) and one line per series
trend_long <- trend %>% select(year, climate_share, circular_share) %>%    # keep the two share columns
  pivot_longer(-year, names_to = "series", values_to = "share_pct")      # one row per year x series
ggplot(trend_long, aes(x = year, y = share_pct, colour = series)) +      # x = year, y = share, one colour per series
  geom_line(linewidth = 1) + geom_point() +      # lines with points
  facet_wrap(~ series, scales = "free_y") +      # separate panels: the two shares differ by an order of magnitude
  theme_minimal(base_size = 13) + theme(legend.position = "none") +      # clean theme, no legend (panel titles say it)
  labs(title = "Share of world output, 2010-2025 (OpenAlex, primary topic)", x = NULL, y = "% of all works")   # labels

**Interpret.** The circular-economy share rises steadily, from 0.45 percent of world output in 2010 to 0.78 percent in 2024 (numbers of 18 Aug 2026): the set grows faster than science as a whole. The climate-economics share (T10471) does not: it slips from 0.053 to 0.036 percent by 2020 and recovers to 0.046 in 2024, a reminder that one topic is a narrow net (climate economics also lives in energy, environmental science and policy topics). In 2025 the denominator jumps (15.1 million works against 10.8 million in 2024), which pulls both shares down: an ingestion artefact of the new data model rather than a change in science. TODO instructor: rerun at rehearsal; if the jump persists, end the chart in 2024 or restrict to `type = "article"`. Read the level with care: a work has one primary topic, so a paper on the economics of recycling filed under an economics topic is not in the numerator, and the denominator counts everything OpenAlex indexes in its core corpus (`corpus=core` is the API default: the 190 million low-metadata "xpac" records are excluded from numerator and denominator alike).

**Exercise.** Restrict numerator and denominator to `type = "article"` and see whether the shares change.

## B2. Geography: who specialises in circular-economy science?

**Concept.** `group_by = "authorships.countries"` counts, for each country, the works with at least one author affiliated there (whole counting: a paper with Italian and German authors counts once for each country). Dividing a country's share of circular-economy works by its share of all works gives the revealed technological advantage index (RTA, the Balassa index of the patent literature): above 1, the country publishes more on the topic than its size predicts. The same logic runs at institution level; OpenAlex institutions carry a `geo` block with the region, so Italian institutions aggregate to Italian regions.

In [ ]:
# Circular-economy works 2020-2025 by country of the authors' institutions (whole counting), and all works by country
ce_country  <- oa_fetch(entity = "works", primary_topic.id = ce_or, publication_year = "2020-2025", group_by = "authorships.countries")   # numerator
all_country <- oa_fetch(entity = "works", publication_year = "2020-2025", group_by = "authorships.countries")                            # denominator
# key is a URL like https://openalex.org/countries/IT: keep the last part as the country code (Stata: substr or regexs)
rta_country <- ce_country %>% transmute(country = sub(".*/", "", key), country_name = key_display_name, ce = count) %>%   # code, name, CE count
  inner_join(all_country %>% transmute(country = sub(".*/", "", key), all = count), by = "country") %>%   # Stata: merge 1:1 country
  mutate(rta = (ce / sum(ce)) / (all / sum(all))) %>%   # share in circular-economy output over share in all output
  arrange(desc(ce))                                     # largest CE producers first
head(rta_country, 10)                                   # top 10 by CE count

In [ ]:
# Top 20 producers plus Italy, bar chart of RTA (Italy in a different colour)
top20 <- rta_country %>% slice_max(ce, n = 20)          # the 20 largest CE producers
if (!"IT" %in% top20$country) top20 <- bind_rows(top20, filter(rta_country, country == "IT"))   # add Italy if it is not in the top 20
ggplot(top20, aes(x = reorder(country_name, rta), y = rta, fill = country == "IT")) +   # countries sorted by RTA, Italy highlighted
  geom_col(show.legend = FALSE) +                        # bars
  geom_hline(yintercept = 1, linetype = "dashed") +      # RTA = 1: no specialisation
  coord_flip() +                                         # horizontal bars
  scale_fill_manual(values = c("grey60", "#2C5F2D")) +   # grey for others, green for Italy
  theme_minimal(base_size = 13) +                        # clean theme
  labs(title = "RTA in circular-economy science, 2020-2025 (top 20 producers + Italy)", x = NULL, y = "RTA (share in CE works / share in all works)")   # labels

### Italian regions

Two calls give the counts by institution (works with at least one Italian institution, grouped by institution id, the 200 largest groups) and one call gives the location of the 200 largest Italian institutions. The reason for building the URL by hand: when a `group_by` has more than 200 groups, `oa_fetch()` pages through all of them (here thousands of institutions, foreign co-author institutions included). We want the 200 largest only, so we read the JSON of one URL with `jsonlite`, which is what `oa_fetch()` does under the hood. A data-quality patch follows: in the June 2026 data most institutions have a city but no region in `geo`, so a small city-to-region table fills the gap.

In [ ]:
# Build the two URLs by hand: same filters as before plus authorships.institutions.country_code:IT, grouped by institution id
key_param <- if (my_key != "PASTE-YOUR-KEY") paste0("&api_key=", my_key) else ""      # the key can travel as a URL parameter
url_ce  <- paste0("https://api.openalex.org/works?filter=authorships.institutions.country_code:IT,publication_year:2020-2025,primary_topic.id:",   # CE works with an Italian institution
                  ce_or, "&group_by=authorships.institutions.id&per-page=200", key_param)                                                        # grouped by institution, first 200 groups
url_all <- paste0("https://api.openalex.org/works?filter=authorships.institutions.country_code:IT,publication_year:2020-2025",                     # all works with an Italian institution
                  "&group_by=authorships.institutions.id&per-page=200", key_param)
# Read the JSON and keep the group_by block: a data frame with key (institution id), key_display_name, count
ce_inst  <- jsonlite::fromJSON(URLencode(url_ce))$group_by     # numerator by institution
all_inst <- jsonlite::fromJSON(URLencode(url_all))$group_by    # denominator by institution
head(all_inst, 5)     # foreign co-author institutions appear too, and some counts look inflated (disambiguation noise in the June 2026 data)

In [ ]:
# Region of the 200 largest Italian institutions (one page of 200, sorted by works_count): the geo block has city, region, country
# openalexR 3.x: sort and paging go inside oa_options(); on 2.0.0 write options = list(sort = "works_count:desc"), per_page = 200, pages = 1 as top-level arguments
inst_it <- oa_fetch(entity = "institutions", country_code = "IT",
                    options = oa_options(sort = "works_count:desc", per_page = 200, pages = 1))
inst_geo <- inst_it %>% select(id, display_name, geo) %>% unnest(geo, names_sep = "_") %>%    # Stata: reshape long on the geo block
  select(id, display_name, geo_city, geo_region)                                              # keep id, name, city, region
head(inst_geo, 5)                 # first five institutions
sum(is.na(inst_geo$geo_region))   # data quality: on 18 Aug 2026 about 130 of the 200 have no region in geo, but all have a city

In [ ]:
# Patch: a city-to-region lookup for the cities of these 200 institutions (region names as OpenAlex spells them)
city_region <- c(
  "Ancona" = "The Marches", "Aviano" = "Friuli Venezia Giulia", "Bari" = "Apulia", "Benevento" = "Campania",
  "Bergamo" = "Lombardy", "Bologna" = "Emilia-Romagna", "Bolzano" = "Trentino-Alto Adige", "Brescia" = "Lombardy",
  "Cagliari" = "Sardinia", "Camerino" = "The Marches", "Campobasso" = "Molise", "Candiolo" = "Piedmont",
  "Caserta" = "Campania", "Cassino" = "Lazio", "Catania" = "Sicily", "Catanzaro" = "Calabria",
  "Chieti" = "Abruzzo", "Enna" = "Sicily", "Ferrara" = "Emilia-Romagna", "Fisciano" = "Campania",
  "Florence" = "Tuscany", "Foggia" = "Apulia", "Frascati" = "Lazio", "Genoa" = "Liguria",
  "Ispra" = "Lombardy", "Lavagna" = "Liguria", "Lecce" = "Apulia", "Legnaro" = "Veneto",
  "L’Aquila" = "Abruzzo", "L'Aquila" = "Abruzzo", "Macerata" = "The Marches", "Meldola" = "Emilia-Romagna",
  "Messina" = "Sicily", "Milan" = "Lombardy", "Modena" = "Emilia-Romagna", "Monserrato" = "Sardinia",
  "Monza" = "Lombardy", "Naples" = "Campania", "Padua" = "Veneto", "Palermo" = "Sicily",
  "Parma" = "Emilia-Romagna", "Pavia" = "Lombardy", "Perugia" = "Umbria", "Pino Torinese" = "Piedmont",
  "Pisa" = "Tuscany", "Potenza" = "Basilicata", "Pozzilli" = "Molise", "Reggio Calabria" = "Calabria",
  "Reggio Emilia" = "Emilia-Romagna", "Rende" = "Calabria", "Rome" = "Lazio", "Rozzano" = "Lombardy",
  "San Donato Milanese" = "Lombardy", "San Giovanni Rotondo" = "Apulia", "San Michele all'Adige" = "Trentino-Alto Adige", "Sassari" = "Sardinia",
  "Sesto Fiorentino" = "Tuscany", "Siena" = "Tuscany", "Teramo" = "Abruzzo", "Trento" = "Trentino-Alto Adige",
  "Trieste" = "Friuli Venezia Giulia", "Turin" = "Piedmont", "Udine" = "Friuli Venezia Giulia", "Urbino" = "The Marches",
  "Varese" = "Lombardy", "Venice" = "Veneto", "Vercelli" = "Piedmont", "Verona" = "Veneto",
  "Viterbo" = "Lazio")
# Keep the region when present, else look it up from the city (Stata: replace region = lookup if missing(region))
inst_geo <- inst_geo %>% mutate(region = coalesce(geo_region, unname(city_region[geo_city])))
sum(is.na(inst_geo$region))       # institutions still without a region (city not in the lookup) are dropped by the joins below

In [ ]:
# Aggregate institution counts to regions (Stata: merge m:1 institution using geo, then collapse (sum) count, by(region))
ce_region <- ce_inst %>% inner_join(inst_geo, by = c("key" = "id")) %>%      # inner join drops foreign and small institutions
  filter(!is.na(region)) %>% group_by(region) %>% summarise(ce = sum(count), .groups = "drop")     # CE count per region (Stata: collapse (sum))
all_region <- all_inst %>% inner_join(inst_geo, by = c("key" = "id")) %>%    # same for all works
  filter(!is.na(region)) %>% group_by(region) %>% summarise(all = sum(count), .groups = "drop")    # total count per region
rta_region <- all_region %>% left_join(ce_region, by = "region") %>%         # regions with no CE institution keep NA (Stata: merge 1:1 region)
  mutate(ce = replace_na(ce, 0L),                        # NA -> 0 CE works
         rta = (ce / sum(ce)) / (all / sum(all))) %>%     # same formula as for countries, Italy = 1
  arrange(desc(rta))                                     # most specialised region first
rta_region                                               # print the table

In [ ]:
# Bar chart of the regional RTA, regions sorted by RTA
ggplot(rta_region, aes(x = reorder(region, rta), y = rta)) +   # x = region ordered by RTA, y = RTA
  geom_col(fill = "#0B7A75") +                           # teal bars
  geom_hline(yintercept = 1, linetype = "dashed") +      # RTA = 1: the Italian average
  coord_flip() +                                         # horizontal bars
  theme_minimal(base_size = 13) +                        # clean theme
  labs(title = "RTA in circular-economy science by Italian region, 2020-2025",                     # labels
       subtitle = "Institution counts summed by region; dashed line = Italian average", x = NULL, y = "RTA")

**Interpret.** China leads the raw counts (78,000 works in 2020-2025 on 18 Aug 2026), then the United States, India, Indonesia and Brazil. The RTA tells a different story: India (1.38), China (1.36), Brazil (1.29) and Indonesia (1.25) are specialised, the United States (0.46), France (0.73), the United Kingdom (0.74) and Germany (0.76) are not; Italy sits at the world average (0.99), the highest of the large western European producers. Waste-management and recycling topics have a large output in emerging economies (local journals, applied engineering), so the topic set decides the answer: state it and keep the id list next to the results. Regional RTAs rest on institution counts: they double count multi-institution papers, they inherit OpenAlex's institution disambiguation errors (a few Italian institutions show implausible totals in the June 2026 data), and they needed a patch because two thirds of the largest Italian institutions have no `region` in their `geo` block, only a city. With that patch, southern and central regions lead (Basilicata, the Marches, Calabria, Sicily, Campania, Molise and Apulia, the region hosting the school, all above 1.6) and Friuli Venezia Giulia, Liguria and Lombardy (0.44) close the ranking; the Aosta Valley has no institution in the top 200 and does not appear. TODO instructor: reread the regional table at rehearsal, the numbers move with each OpenAlex update. Compare with the BigQuery answer in leg C, which counts every work without the 200-group cap.

**Exercise.** Swap the topic set for `"T10471"` and redo the country RTA for climate economics.

## B3. SDG landscape: what is Italy's SDG research made of?

**Concept.** OpenAlex tags works with UN Sustainable Development Goals (`sustainable_development_goals`, id `https://openalex.org/sdgs/13` for climate action) with a classifier trained on the Aurora SDG queries. `group_by = "sustainable_development_goals.id"` for one country gives its SDG profile in one call (the OpenAlex recipe "Map SDG research", help.openalex.org/tutorials/map-sdg-research/); two countries side by side show different priorities; filtering on one SDG and grouping by subfield shows what a goal is made of. Caveat: SDG tags are classifier output, and databases and classifiers disagree on which papers count (Kashnitsky et al. 2024, *Quantitative Science Studies* 5(2) 408-425, doi:10.1162/qss_a_00304; Ottaviani and Stahlschmidt 2024, arXiv:2405.03007).

In [ ]:
# SDG counts for Italy and Germany, 2020-2025 (Stata: two collapse (count), by(sdg), then append)
sdg_it <- oa_fetch(entity = "works", authorships.countries = "IT", publication_year = "2020-2025", group_by = "sustainable_development_goals.id")
sdg_de <- oa_fetch(entity = "works", authorships.countries = "DE", publication_year = "2020-2025", group_by = "sustainable_development_goals.id")   # TODO instructor: pick the partner country
# Stack the two tables and express counts as a share of the country's total works (totals come from all_country in B2)
sdg <- bind_rows(mutate(sdg_it, country = "IT"), mutate(sdg_de, country = "DE")) %>%   # stack the two tables (Stata: append)
  mutate(sdg = paste0("SDG ", sub(".*/", "", key)), goal = key_display_name) %>%      # "SDG 13" from the URL key, goal name
  left_join(all_country %>% transmute(country = sub(".*/", "", key), total = count), by = "country") %>%   # each country's total works (Stata: merge m:1)
  mutate(share_pct = 100 * count / total)                # share of the country's works carrying the tag
sdg %>% select(country, sdg, goal, count, share_pct) %>% arrange(country, desc(share_pct)) %>% head(10)   # top SDGs per country

In [ ]:
# Paired bar chart: SDGs in numeric order, one bar per country
sdg <- sdg %>% mutate(sdg = factor(sdg, levels = paste0("SDG ", 1:17)))   # factor so the x axis runs SDG 1 to SDG 17
ggplot(sdg, aes(x = sdg, y = share_pct, fill = country)) +   # x = SDG, y = share, one colour per country
  geom_col(position = "dodge") +                             # side-by-side bars
  scale_fill_manual(values = c(IT = "#2C5F2D", DE = "#D4A843")) +   # green Italy, gold Germany
  theme_minimal(base_size = 13) +                            # clean theme
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) + # tilted x labels
  labs(title = "SDG profile of national output, 2020-2025", x = NULL, y = "% of the country's works", fill = NULL)   # labels

In [ ]:
# What is Italy's SDG 13 (climate action) research made of? Filter on the SDG and group by subfield of the primary topic
sdg13_sub <- oa_fetch(entity = "works", authorships.countries = "IT",
                      sustainable_development_goals.id = "https://openalex.org/sdgs/13",
                      publication_year = "2020-2025", group_by = "primary_topic.subfield.id")
# The subfield list has more than 200 groups, so openalexR fetches two pages; keep the ten largest (Stata: gsort -count, list in 1/10)
sdg13_sub %>% arrange(desc(count)) %>% mutate(share_pct = 100 * count / sum(count)) %>%
  select(key_display_name, count, share_pct) %>% head(10)

**Interpret.** SDG 3 (health) dominates every country's profile because biomedical output is large (about 12 percent of German and 17 percent of Italian works carry the tag); the reading is relative: where Italy's bar exceeds Germany's and where it falls short (TODO instructor: name the two or three goals from the chart at rehearsal). Italy's SDG 13 output is led by Global and Planetary Change (12 percent) and Atmospheric Science (7 percent); the policy and social science subfields (Management, Monitoring, Policy and Law; Sociology and Political Science; Economics and Econometrics) follow with 3 to 4 percent each, ahead of most engineering subfields. Confraria, Ciarli and Noyons (2024, *Research Policy* 53(3) 104950) read such profiles as research priorities; here they are also a property of the classifier: repeat the profile with the Scopus or Web of Science SDG filters and the ranking changes, so the caveat above is not a footnote.

**Exercise.** Replace SDG 13 with SDG 12 (responsible consumption and production) and compare the subfield mix with the circular-economy topic set of B1.

## B4. Extras (not demoed): counting, snowballing, sampling, exporting

**Concept.** Four things you will need in your own work: count before you download; build a citation neighbourhood around a paper (the works it cites and the works citing it, with a filter on the citing side to keep the volume down); draw a random sample of a large set instead of downloading it all (reproducible thanks to a seed); write the result to CSV or Stata. In openalexR: `count_only = TRUE`, `oa_snowball()`, `options = oa_options(sample = 200, seed = 42)`, `haven::write_dta()`.

In [ ]:
# Count before you download: how many circular-economy articles 2020-2025? (Stata: count if ...)
n_ce <- oa_fetch(entity = "works", primary_topic.id = ce_or, publication_year = "2020-2025", type = "article", count_only = TRUE)   # meta block only
n_ce$count   # the number of matching articles (295,333 on 18 Aug 2026)

In [ ]:
# Snowball around one paper: the works it cites (189 references, 172 of them OpenAlex works) and the works citing it since June 2026 (restricted to keep the download small)
sb <- oa_snowball(identifier = "W1732240353", citing_params = list(from_publication_date = "2026-06-01"), verbose = TRUE)
sb$nodes %>% count(oa_input)     # oa_input = TRUE marks the input paper; the rest are cited or citing works
head(sb$edges)                   # from cites to

In [ ]:
# Random sample of 200 circular-economy articles 2020-2025 (Stata: set seed 42, sample 200, count); the seed makes it reproducible
# openalexR 3.x: sample and seed go in oa_options(); paging = "page" because the API refuses cursor paging with sample
# openalexR 2.0.0: options = list(sample = 200, seed = 42) and paging = "page" as a top-level argument
ce_sample <- oa_fetch(entity = "works", primary_topic.id = ce_or, publication_year = "2020-2025", type = "article",   # same filter as the count above
                      options = oa_options(sample = 200, seed = 42, paging = "page"))                                 # 200 random works, fixed seed
summary(ce_sample$cited_by_count)         # skewed as always: median small, mean pulled up by a few papers
ggplot(ce_sample, aes(x = cited_by_count + 1)) + geom_histogram(bins = 30, fill = "#2C5F2D") + scale_x_log10() +   # histogram on a log axis (+1 keeps the zeros)
  theme_minimal(base_size = 13) +          # clean theme
  labs(title = "Citations of 200 random circular-economy articles, 2020-2025", x = "cited_by_count + 1 (log scale)", y = "articles")   # labels

In [ ]:
# Export to CSV and Stata (Stata: export delimited, save); in Colab, download from the Files pane on the left
write.csv(rta_country, "rta_country_2020_2025.csv", row.names = FALSE)                    # CSV in the working directory
if (!requireNamespace("haven", quietly = TRUE)) install.packages("haven", quiet = TRUE)   # haven is part of the tidyverse, present in Colab
haven::write_dta(rta_country, "rta_country_2020_2025.dta")                                # Stata .dta (column names are valid Stata names)
list.files(pattern = "rta_country")                                                       # the two files just written

**Interpret.** Counting first is the habit that keeps a project inside its budget: a count costs 0.0001 USD, a download of 100,000 works costs 500 calls of 200 records and minutes of waiting. Snowballing is how a literature review or a knowledge-flow measure starts; sampling is how you look at a population you cannot download in a session; the export is what enters your Stata do-file.

**Exercise.** Snowball around your own most-cited paper and count how many citing works carry an SDG tag.

## Where the same numbers come from elsewhere

Leg C (BigQuery, `subugoe-collaborative.openalex_walden.works`, June 2026 snapshot) recomputes B1 and B2 with SQL over every record: no 200-group cap, no paging, and the `is_xpac` flag replaces the API's `corpus=core` default. Expect small differences from snapshot dates and from the group cap. See `notebook/README.md` for the mapping between the R and Python versions.

## Build notes (for the instructor)

Verified on 18 Aug 2026, keyless, from R 4.4.2 with openalexR 3.1.0 (single-work fetch, group_by by year, institutions fetch, `oa_options()`) and openalexR 2.0.0 (group_by by country, SDG, subfield):

- `oa_fetch(entity = "works", identifier = "W1732240353")`: 43 columns; `sustainable_development_goals` is NA; topics 12 rows (3 topics x 4 levels); 3 authorships, 4 affiliation rows.
- T10471 by year 2010-2025: 16 groups; 2023 = 4,864; 2024 = 4,994; 2025 = 4,809; T10471 total 104,773.
- IT 2020-2025 by SDG: 17 groups; SDG 3 = 227,082; SDG 13 = 24,368; SDG 12 = 16,481.
- IT + SDG 13 by `primary_topic.subfield.id`: 236 groups (two pages).
- Works with an Italian institution 2020-2025: 1,347,612; the group_by by institution id returns 200 groups per page and pages with `cursor=*`, which is why the region step reads the URL directly (checked with `jsonlite::fromJSON()`: 200 rows).
- `oa_fetch(entity = "institutions", country_code = "IT", options = oa_options(sort = "works_count:desc", per_page = 200, pages = 1))`: 200 rows in one page; 130 of them without `geo$region`, hence the city lookup (the lookup covers every city in that list on 18 Aug 2026; the aggregation cell was checked offline on the downloaded table).
- The R checks of the CE-dependent calls (country group_by: 197 groups; `count_only`; `oa_options(sample = 200, seed = 42, paging = "page")`: 200 works) ran with an earlier 9-topic list; the final 28-topic list was run end to end in the Python twin (same API calls): CE works 2020-2025 by country CN 78,298, US 33,220, IN 29,040, ID 25,805, BR 16,055, IT 11,521 (RTA 0.99); CE articles 2020-2025: 295,333; sample of 200: median 6 citations, max 628; regions: Basilicata 2.54, the Marches 2.48, Calabria 1.99, Sicily 1.97, Campania 1.92, Apulia 1.64, Lazio 1.11, Lombardy 0.44.
- `per-page=200` accepted by the API on that date.
- The CE-dependent calls pass `ce_or` (the ids joined by `|` in one string) instead of the character vector: `oa_query()` builds the identical URL for both forms (checked offline against openalexR 3.1.0 on 18 Aug 2026), and the string form does not trigger openalexR's 50-value batching, which would duplicate `group_by` keys if the set grew past 50 ids.
- Not run at build time: `oa_snowball()` (call volume). Check at rehearsal.

openalexR 2.0.0 fails on `oa_fetch(entity = "works", ...)` with "Column name `id` must not be duplicated" (pre-Walden parser): install 3.x.